In [1]:
from ortools.sat.python import cp_model
import numpy as np

In [2]:
wgraph = np.array([[0 for i in range(20)] for j in range(20)])
# Construction du graphe d'adjacence
for i, j in [
    (1, 2), (1, 7),
    (2, 3), (2, 9),
    (3, 4), (3, 10),
    (4, 5), (4, 11), (4, 12),
    (5, 6), (5, 13),
    (6, 14),
    (7, 8), (7, 9),
    (8, 9), (8, 15),
    (9, 10), (9, 16),
    (10, 11), (10, 17),
    (11, 12), (11, 18),
    (12, 13), (12,18),
    (13, 14),
    (13, 19),
    (14, 20),
    (15, 16),
    (16, 17),
    (17, 18),
    (18, 19),
    (19, 20)
]:
    wgraph[i-1][j-1] = 1
    wgraph[j-1][i-1] = 1

# Production des porteurs d'eau
cart_prod = (2, 1)
MAX_INT = 10000

In [3]:
# Besoins des quartiers
wneeds = [1, 1, 1, 1, 1, 1,
          1,
          1, 1, 1, 1, 1, 1, 1,
          2, 1, 1, 1, 1, 1]
# Nombre de porteurs d'eau
ncarts = 3

In [4]:
m = cp_model.CpModel()
# Nombre de porteurs d'eau par quartier
wcarts = [ m.NewIntVar(0, ncarts, name=f'carts{i+1}') for i in range(20) ]
# Points gagnés (ou perdus) par quartier
wpts = [ m.NewIntVar(-2, 1, name=f'pts{i+1}') for i in range(20) ]
# Surplus ou déficit d'eau par quartier
wqty = [ m.NewIntVar(-MAX_INT, MAX_INT, name=f'water{i+1}') for i in range(20) ]
# Variable binaire indiquant un quartier en surplus (+1 point)
wpos = [ m.NewBoolVar(name=f'pos{i+1}') for i in range(20) ]
# Variable binaire indiquant un quartier en léger déficit (+0 points)
wnil = [ m.NewBoolVar(name=f'nil{i+1}') for i in range(20) ]
# Variable binaire indiquant un quartier en déficit (-2 points)
wneg = [ m.NewBoolVar(name=f'neg{i+1}') for i in range(20) ]
for i in range(20):
    # La quantité d'eau est celle fournie par les porteurs
    m.Add(wqty[i] == wcarts[i] * cart_prod[0] + sum(wgraph[i, :] * wcarts) * cart_prod[1] - wneeds[i])
    # Les points peuvent être positifs si la quantité d'eau est >= 0
    m.Add(wqty[i] >= 0).OnlyEnforceIf(wpos[i])
    # Les points peuvent être nuls si la quantité d'eau est == -1
    m.Add(wqty[i] == -1).OnlyEnforceIf(wnil[i])
    # Les points peuvent être négatifs si la quantité d'eau est <= -2
    m.Add(wqty[i] <= -2).OnlyEnforceIf(wneg[i])
    # Par quartier, les points doivent être positifs, négatifs ou nuls
    m.Add(wpos[i] + wnil[i] + wneg[i] == 1)
    # Somme des points par quartier
    m.Add(wpts[i] == wpos[i] - 2 * wneg[i])
# Contrainte sur le nombre total de porteurs
m.Add(sum(wcarts) <= ncarts)

# On cherche à maximiser nos points
m.Maximize(sum(wpts))

solver = cp_model.CpSolver()
status = solver.Solve(m)

# Affichage: nombre de porteurs par quartier
print(solver.StatusName(status))
for tab in (wcarts,):
    print([solver.Value(tab[i]) for i in range(6)])
    print([solver.Value(tab[i]) for i in range(6, 7)])
    print([solver.Value(tab[i]) for i in range(7, 14)])
    print([solver.Value(tab[i]) for i in range(14, 20)])
    print()

OPTIMAL
[0, 0, 0, 0, 0, 0]
[0]
[0, 0, 1, 0, 0, 1, 0]
[1, 0, 0, 0, 0, 0]

